Cration of a Table sourcing from csv files.

In [0]:
source_directory = "/Volumes/workspace_file_loading/staging_file_csv/vendite_trials/"
checkpoint_path = "/Volumes/workspace_file_loading/staging_file_csv/vendite_trials/_checkpoints/vendite_info"
target_table = "workspace.staging.vendite_info"

df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
    .load(source_directory))

query = (df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table))

query.awaitTermination()

CODE DESCRIPTION

- source_directory: path volume where we store our files.
- checkpoint_path: it is an hidden folder used by databricks to keep tracking files already read and the current schema
- target_table: final table path (catalog-schema-table name)

FILE READING:
- **spark.readStream:** it starts reading with a streaming mode
- **.format("cloudFiles")**: it allows spark to use the autoloader mode.
- **.option("cloudFiles.format", "csv"):** it specifies the format of source files
- **.option("header", "true"):** it sets the first row as header
- **.option("cloudFiles.schemaEvolutionMode", "addNewColumns"):** if a new file has new columns, those are added within the target table
- **.option("cloudFiles.schemaLocation", ...):** it stores the structure of the actual schema within the specified path. This is used to make comparison with future files
- **.load(source_directory):** it loades tha file within the table as a structured data.

FILE LOADING:
- **.format("delta"):** table destination set as a delta lake table
- **.option("checkpointLocation", checkpoint_path):** Key point regarding idempotence: the processing status is recorded here to ensure that old files are not reprocessed in subsequent runs.
- **.outputMode("append"):** new data is added/appended at the current table
- **.option("mergeSchema", "true"):** the table schema is updated if new columns are added
- **.trigger(availableNow=True):** it tells databricks to read the files, load them and then to shut down
- **.toTable(target_table):** destination of the data.
- **query.awaitTermination()**